In [1]:
from __future__ import print_function

import os


os.chdir("../")

In [2]:
from hyperopt import STATUS_OK, Trials, tpe
from hyperas.distributions import choice, uniform, loguniform
from hyperas import optim

import tensorflow as tf
from tensorflow.keras import models, optimizers

import numpy as np

from autoencoder.variational_autoencoder import VariationalAutoencoder
from common.dataloader import load_cifar10
from common.model import get_callbacks, get_compile_args
from common.utils import (hyperas_path, i, best_score, save_logs, 
                        init, load_samples)

In [3]:
def create_data():
    init()

    feature_train0, y_train, *_ = load_cifar10(return_features=True, onehot_labels=True, verbose=0)
    feature_train1, *_ = load_cifar10(return_features=True, preprocess="normalize", verbose=0)
    feature_train2, *_ = load_cifar10(return_features=True, preprocess="min-max", verbose=0)

    clf0 = models.load_model("./models/hyperas/cifar10_dnn_model_00.h5")
    clf1 = models.load_model("./models/hyperas/cifar10_dnn_model_00B.h5")
    clf2 = models.load_model("./models/hyperas/cifar10_dnn_model_00C.h5")

    return feature_train0, feature_train1, feature_train2, y_train, clf0, clf1, clf2

In [4]:
def create_model(feature_train0, feature_train1, feature_train2, y_train, clf0, clf1, clf2):
    global best_score, i  


    hidden_num = {{choice([1, 2, 3, 4])}}
    latent_dim = {{choice([8, 16, 32, 64, 128])}}
    units4 = latent_dim * {{choice([1, 2, 4])}}
    units3 = units4 * {{choice([1, 2, 4])}}
    units2 = units3 * {{choice([1, 2, 4])}}
    units1 = units2 * {{choice([1, 2, 4])}}

    actv = {{choice(["relu", "elu", "selu", "prelu", "tanh", "sigmoid"])}}
    last_actv = {{choice(["linear", "tanh", "sigmoid"])}}
    use_batch_norm = {{choice([True, False])}}

    distribution = {{choice(["normal", "uniform"])}}
    initialization = {{choice(["glorot", "he", "lecun"])}}
    kernel_init = f"{initialization}_{distribution}"

    lr = {{choice([0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    momentum = {{choice([0.0, 0.01, 0.03, 0.1, 0.3, 0.5, 0.8, 0.9, 0.99])}}
    weight_decay = {{choice([0.0, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    opt_choice = {{choice(["sgd", "rmsprop", "adam", "nadam", "adamax", "adamw"])}}

    loss_choice = {{choice(["mae", "mse"])}}

    beta = {{choice([0.1, 0.25, 0.5, 0.75, 1., 1.25, 1.5, 1.75, 2.])}}


    if opt_choice == "sgd":
        optimizer = optimizers.SGD(learning_rate=lr, momentum=momentum, decay=weight_decay, nesterov=True)
    elif opt_choice == "rmsprop":
        optimizer = optimizers.RMSprop(learning_rate=lr, momentum=momentum, decay=weight_decay)
    elif opt_choice == "adam":
        optimizer = optimizers.Adam(learning_rate=lr, decay=weight_decay)
    elif opt_choice == "nadam":
        optimizer = optimizers.Nadam(learning_rate=lr, decay=weight_decay)
    elif opt_choice == "adamax":
        optimizer = optimizers.Adamax(learning_rate=lr, decay=weight_decay)
    elif opt_choice == "adamw":
        optimizer = optimizers.experimental.AdamW(learning_rate=lr, weight_decay=weight_decay)

    if loss_choice == "bce":
        loss_fn = "binary_crossentropy"
    elif loss_choice == "mae":
        loss_fn = "mean_absolute_error"
    elif loss_choice == "mse":
        loss_fn = "mean_squared_error"

    if last_actv == "linear":
        feature_train = feature_train0
        clf = clf0
    elif last_actv == "tanh":
        feature_train = feature_train1
        clf = clf1
    elif last_actv == "sigmoid":
        feature_train = feature_train2
        clf = clf2


    model = VariationalAutoencoder(
        conditioned=True, 
        class_num=10,
        latent_dim=latent_dim,
        hiddens_dims=[units1, units2, units3, units4][:hidden_num],
        hiddens_kwargs={
            "actv": actv,
            "use_batch_norm": use_batch_norm,
            "kernel_init": kernel_init
        },
        last_activation=last_actv, 
        beta=beta,
        compile_args=get_compile_args(
            optimizer=optimizer,
            loss=loss_fn
        ),
        name="cifar10_vae_01"
    )

    save_logs(model.name, i, 
            search_space=[hidden_num, units1, units2, units3, units4, latent_dim, actv, last_actv, 
                        use_batch_norm, distribution, initialization, kernel_init, lr, momentum, 
                        weight_decay, opt_choice, loss_choice, beta], 
            names=["hidden_num", "units1", "units2", "units3", "units4", "latent_dim", "actv", "last_actv", 
                "use_batch_norm", "distribution", "initialization", "kernel_init", "lr", "momentum", 
                "weight_decay", "opt_choice", "loss_choice", "beta"])


    history = model.train(
        feature_train, y_train, 
        train_num=-1, 
        epochs=10, 
        batch_size=512, 
        callbacks_list=get_callbacks(monitor="decoder_accuracy", verbose=0), 
        clf=clf, 
        verbose=0
    )

    val_acc = max(history["decoder_accuracy"])

    trainable_num = np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    trainable_num /= 10_000_000

    x_gen, _ = model.generate()
    std_list = []
    for j in range(10):
        class_i_std_mean = x_gen[j*500: (j+1)*500].std(axis=0).mean()
        std_list.append(class_i_std_mean)
    mean_of_stds = 1 * np.mean(std_list)

    score = val_acc - trainable_num + mean_of_stds

    if score > best_score:
        best_score = score
        model.save(os.path.join(hyperas_path, f"tf/{model.name}.tf"))

    save_logs(model.name, i, metrics={
            "val_acc": val_acc,
            "trainable_num": trainable_num,
            "mean_of_stds": mean_of_stds,
            "score": score,
            "best_score": best_score})

    i += 1


    return {"loss": -score, "status": STATUS_OK, "model": None}

In [5]:
best_run, _ = optim.minimize(
    model=create_model,
    data=create_data,
    algo=tpe.suggest,
    max_evals=200,
    trials=Trials(),
    notebook_name="notebooks/cifar10 hp-tuning vae"
)

>>> Imports:
#coding=utf-8

from __future__ import print_function

try:
    import os
except:
    pass

try:
    from hyperopt import STATUS_OK, Trials, tpe
except:
    pass

try:
    from hyperas.distributions import choice, uniform, loguniform
except:
    pass

try:
    from hyperas import optim
except:
    pass

try:
    import tensorflow as tf
except:
    pass

try:
    from tensorflow.keras import models, optimizers
except:
    pass

try:
    import numpy as np
except:
    pass

try:
    from autoencoder.variational_autoencoder import VariationalAutoencoder
except:
    pass

try:
    from common.utils import get_compile_args, get_callbacks, hyperas_path, i, best_score, save_logs, init, load_samples, load_cifar10
except:
    pass

try:
    from sklearn.metrics import classification_report
except:
    pass

>>> Hyperas search space:

def get_space():
    return {
        'hidden_num': hp.choice('hidden_num', [1, 2, 3, 4]),
        'latent_dim': hp.choice('latent_dim', [8, 16, 32, 64

In [6]:
best_run

{'actv': 2,
 'beta': 1,
 'distribution': 0,
 'hidden_num': 0,
 'initialization': 1,
 'last_actv': 1,
 'latent_dim': 0,
 'latent_dim_1': 0,
 'latent_dim_2': 1,
 'latent_dim_3': 0,
 'latent_dim_4': 0,
 'loss_choice': 1,
 'lr': 6,
 'momentum': 7,
 'opt_choice': 3,
 'use_batch_norm': 0,
 'weight_decay': 0}

In [7]:
from sklearn.metrics import classification_report


*_, clf0, clf1, clf2 = create_data()
model = models.load_model(os.path.join(hyperas_path, "tf/cifar10_vae_01.tf"))
model.summary()

vae = VariationalAutoencoder(
    conditioned=True, 
    class_num=10,
    latent_dim=8,
    hiddens_dims=[16],
    hiddens_kwargs={
        "actv": "selu",
        "use_batch_norm": True,
        "kernel_init": "he_normal"
    },
    last_activation="tanh", 
    beta=0.25,
    compile_args=get_compile_args(
        optimizer=optimizers.Nadam(learning_rate=0.1, decay=0.),
        loss="mean_squared_error"
    ),
    name="cifar10_vae_00"
)

vae.set_weights(model.get_weights())

x_gen, y_true = vae.generate(classes=list(range(10)), onehot_labels=False)
y_preds = clf1.predict(x_gen)
y_preds = np.argmax(y_preds, axis=-1)
print(classification_report(y_true, y_preds, digits=4))

Model: "cifar10_vae_01"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 encoder (Functional)        [(None, 8),               33264     
                              (None, 8),                         
                              (None, 8)]                         
                                                                 
 decoder (Functional)        (None, 2048)              35168     
                                                                 
Total params: 68,438
Trainable params: 68,368
Non-trainable params: 70
_________________________________________________________________
157/157 [==============================] - 0s 2ms/step
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       500
           1     1.0000    1.0000    1.0000       500
           2     1.0000    1.0000    1.0000       500
           3     1.0000    1.0000    1.